In [1]:
# %% [markdown]
# # Chile Mineral Supply Chain: Data Pipeline
#
# Downloads and harmonises facility data from Sernageomin (ArcGIS) and USGS,
# infers mine-to-plant links by commodity and proximity, cross-references with
# HS92 trade data and COCHILCO production statistics.
#
# **Inputs:**
# - ArcGIS FeatureServer (Sernageomin, downloaded live)
# - `MINFAC_LAC.csv` (USGS, local)
# - `Chile_HS92_Exports_6digit.csv` (Harvard Growth Lab, local)
# - `COCHILCO_Production_2005_2024.xlsx` (extracted yearbook, local)
#
# **Outputs (all to `Preliminary/`):**
# - `Chile_Minerals_Inventory.csv` / `.geojson`
# - `Chile_Mine_Plant_Links.csv`
# - `Chile_Supply_Chain_Summary.csv`

# %% 0. Setup
import os, json, time, warnings, requests
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
USGS_PATH = "/Users/leoss/Downloads/MINFAC_LAC.csv"
HS92_PATH = os.path.join(BASE_DIR, "data", "hs92_country_product_year_6.csv")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

DIR_OUT    = os.path.join(BASE_DIR, "Outputs")
DIR_HTML   = os.path.join(BASE_DIR, "Interactive")
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
for d in [DIR_OUT, DIR_HTML, DIR_PRELIM]:
    os.makedirs(d, exist_ok=True)

BASE_URL = "https://services1.arcgis.com/OyjvVdFTl5hfSdX3/arcgis/rest/services/EMC_F/FeatureServer"
CHILE_LAT_MIN, CHILE_LAT_MAX = -56.0, -17.5
CHILE_LON_MIN, CHILE_LON_MAX = -76.0, -66.0
LINK_MAX_KM = 150

start_time = time.time()

# %% 1. Commodity harmonisation and helper functions
COMMODITY_MAP = {
    "Cobre": "Copper", "Oro": "Gold", "Plata": "Silver",
    "Hierro": "Iron", "Litio": "Lithium", "Nitrato": "Nitrate",
    "Boro": "Boron", "Cinc": "Zinc", "Manganeso": "Manganese",
    "Titanio": "Titanium", "Tierras raras": "Rare Earths",
    "Molibdeno": "Molybdenum", "Cobalto": "Cobalt", "Renio": "Rhenium",
    "Potasio": "Potassium", "Yodo": "Iodine", "Selenio": "Selenium",
    "Telurio": "Tellurium", "Antimonio": "Antimony",
    "Nitrates": "Nitrate", "Iron and steel": "Iron",
    "Sulfuric acid": "Sulfuric Acid", "Calcium carbonate": "Calcium Carbonate",
    "Phosphatic materials": "Phosphate", "Sodium sulfate": "Sodium Sulfate",
    "Potassium chloride": "Potassium", "Natural gas": "Natural Gas",
}

def harmonize_commodity(name):
    if pd.isna(name):
        return name
    name = str(name).strip()
    return COMMODITY_MAP.get(name, name)

def parse_commodity_list(combo_string):
    if pd.isna(combo_string):
        return []
    parts = str(combo_string).replace(",", "-").replace("/", "-").split("-")
    return [harmonize_commodity(p.strip()) for p in parts if p.strip()]

def classify_usgs_plant(name):
    name = str(name).lower()
    for kw, label in [("smelter", "Smelter"), ("refin", "Refinery"),
                       ("sx-ew", "SX-EW Plant"), ("sx/ew", "SX-EW Plant"),
                       ("concentrat", "Concentrator"), ("flotation", "Concentrator"),
                       ("milling", "Concentrator"), ("pellet", "Pellet Plant"),
                       ("grinding", "Grinding Plant"), ("steel", "Steel Plant")]:
        if kw in name:
            return label
    return "Processing Plant"

USGS_STATUS_MAP = {"A": "Active", "C": "Closed", "S": "Standby", "CM": "Care & Maintenance"}

FACILITY_STAGE = {
    "Mine (active)": "extraction", "Mine (idle)": "extraction",
    "Prospect/Project": "extraction", "Mine (USGS)": "extraction",
    "Concentrator": "processing", "SX-EW Plant": "processing",
    "Smelter": "processing", "Refinery": "processing",
    "Processing Plant": "processing", "Pellet Plant": "processing",
    "Grinding Plant": "processing", "Steel Plant": "processing",
}

def download_layer(layer_id, with_geometry=True):
    all_features, offset = [], 0
    while True:
        params = {"where": "1=1", "outFields": "*", "f": "json",
                  "resultRecordCount": 2000, "resultOffset": offset}
        if with_geometry:
            params.update({"outSR": "4326", "returnGeometry": "true"})
        r = requests.get(f"{BASE_URL}/{layer_id}/query", params=params, timeout=60)
        r.raise_for_status()
        features = r.json().get("features", [])
        if not features:
            break
        all_features.extend(features)
        if len(features) < 2000:
            break
        offset += len(features)
    return all_features

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# %% 2. Download Sernageomin critical minerals
dep_features = download_layer(0, with_geometry=True)
res_features = download_layer(3, with_geometry=False)
print(f"Deposits: {len(dep_features)}, Reserve/resource records: {len(res_features)}")

rows = []
for feat in dep_features:
    row = feat.get("attributes", {})
    geom = feat.get("geometry")
    if geom and "x" in geom:
        row["geometry"] = Point(geom["x"], geom["y"])
        row["LATITUD"] = geom["y"]
        row["LONGITUD"] = geom["x"]
    else:
        row["geometry"] = None
    rows.append(row)
serna_gdf = gpd.GeoDataFrame(pd.DataFrame(rows), geometry="geometry", crs="EPSG:4326")

reservas = pd.DataFrame([f["attributes"] for f in res_features])
if len(reservas) > 0:
    rp = reservas.pivot_table(index="ID_DEPOSITO", columns=["MINERAL", "TIPO"],
                              values="VALOR", aggfunc="sum")
    rp.columns = [f"{m}_{t}" for m, t in rp.columns]
    rp = rp.reset_index()
    serna_gdf = serna_gdf.merge(rp, left_on="ID", right_on="ID_DEPOSITO", how="left")
    serna_gdf.drop(columns=["ID_DEPOSITO"], errors="ignore", inplace=True)

serna_gdf["SOURCE"] = "Sernageomin_2025"
serna_gdf["STATUS"] = serna_gdf["ESTADO_DEPOSITO"].map({
    "Mina en producci\u00f3n": "Active", "Mina paralizada": "Idle",
    "Mina Paralizada": "Idle", "Prospecto/Proyecto": "Prospect/Project"})
serna_gdf["FACILITY_TYPE"] = serna_gdf["ESTADO_DEPOSITO"].map({
    "Mina en producci\u00f3n": "Mine (active)", "Mina paralizada": "Mine (idle)",
    "Mina Paralizada": "Mine (idle)", "Prospecto/Proyecto": "Prospect/Project"})
serna_gdf["FACILITY_NAME"] = serna_gdf["NOMBRE_DEPOSITO"]
serna_gdf["PRIMARY_COMMODITY"] = serna_gdf["CRITICO_1"].apply(harmonize_commodity)
serna_gdf["ALL_COMMODITIES_RAW"] = serna_gdf["COMBINACION_CRITICO"]
for col in ["OPERATOR_NAME", "OWNER_NAME", "CAPACITY", "CAPACITY_UNITS"]:
    serna_gdf[col] = None

rename = {}
for col in serna_gdf.columns:
    new = col
    for es, en in COMMODITY_MAP.items():
        if es in new:
            new = new.replace(es, en)
    new = new.replace("_Recurso", "_Resource").replace("_Reserva", "_Reserve")
    if new != col:
        rename[col] = new
serna_gdf.rename(columns=rename, inplace=True)

print(f"Sernageomin: {serna_gdf.shape}")

# %% 3. Load USGS mineral facilities
usgs = pd.read_csv(USGS_PATH, encoding="latin-1")
chile_usgs = usgs[usgs["COUNTRY"].str.strip().str.lower() == "chile"].copy()

plants = chile_usgs[chile_usgs["FACTYPE"].isin(["P", "B", "OR", "MP", "p"])].copy()
plants["FACILITY_TYPE"] = plants["LOCNAME"].apply(classify_usgs_plant)

usgs_mines = chile_usgs[chile_usgs["FACTYPE"].isin(["M", "m", "DM", "SM"])].copy()
usgs_mines["FACILITY_TYPE"] = "Mine (USGS)"

plants["geometry"] = plants.apply(
    lambda r: Point(r["DDLONG"], r["DDLAT"])
    if pd.notna(r["DDLAT"]) and pd.notna(r["DDLONG"]) else None, axis=1)
plants_gdf = gpd.GeoDataFrame(plants, geometry="geometry", crs="EPSG:4326")

plants_gdf["SOURCE"] = "USGS_MINFAC_2017"
plants_gdf["FACILITY_NAME"] = plants_gdf["LOCNAME"]
plants_gdf["PRIMARY_COMMODITY"] = plants_gdf["COMMODITY"].apply(harmonize_commodity)
plants_gdf["ALL_COMMODITIES_RAW"] = plants_gdf["COMMODITY"]
plants_gdf["OPERATOR_NAME"] = plants_gdf["OPERATOR"]
plants_gdf["OWNER_NAME"] = plants_gdf["OWNER"]
plants_gdf["CAPACITY"] = plants_gdf["ANNCAP"]
plants_gdf["CAPACITY_UNITS"] = plants_gdf["UNITS"]
plants_gdf["LATITUD"] = plants_gdf["DDLAT"]
plants_gdf["LONGITUD"] = plants_gdf["DDLONG"]
plants_gdf["REGION"] = plants_gdf["ADM1"]
plants_gdf["STATUS"] = plants_gdf["STATUS"].map(USGS_STATUS_MAP).fillna(plants_gdf["STATUS"])

usgs_mines_df = usgs_mines[["LOCNAME", "COMMODITY", "OPERATOR", "OWNER", "DDLAT", "DDLONG"]].copy()
usgs_mines_df["LOCNAME_lower"] = usgs_mines_df["LOCNAME"].str.lower().str.strip()

print(f"USGS Chile: {len(chile_usgs)} total, {len(plants)} plants, {len(usgs_mines)} mines")

# %% 4. Merge and deduplicate
all_cols = list(set(serna_gdf.columns) | set(plants_gdf.columns))
for col in all_cols:
    if col not in serna_gdf.columns:
        serna_gdf[col] = None
    if col not in plants_gdf.columns:
        plants_gdf[col] = None

combined = gpd.GeoDataFrame(
    pd.concat([serna_gdf[all_cols], plants_gdf[all_cols]], ignore_index=True), crs="EPSG:4326")
print(f"Before dedup: {len(combined)}")

serna_mask = combined["SOURCE"] == "Sernageomin_2025"
serna_rows = combined[serna_mask].copy()
serna_rows["name_lower"] = serna_rows["FACILITY_NAME"].str.lower().str.strip()

enriched = 0
for idx, row in serna_rows.iterrows():
    matches = usgs_mines_df[usgs_mines_df["LOCNAME_lower"].str.contains(
        row["name_lower"][:8], case=False, na=False, regex=False)]
    if len(matches) > 0:
        best = matches.iloc[0]
        if pd.isna(combined.at[idx, "OPERATOR_NAME"]) and pd.notna(best["OPERATOR"]):
            combined.at[idx, "OPERATOR_NAME"] = best["OPERATOR"]
            enriched += 1
        if pd.isna(combined.at[idx, "OWNER_NAME"]) and pd.notna(best["OWNER"]):
            combined.at[idx, "OWNER_NAME"] = best["OWNER"]
print(f"Enriched {enriched} records with USGS ownership")

combined["_name_lower"] = combined["FACILITY_NAME"].str.lower().str.strip()
combined["_stage"] = combined["FACILITY_TYPE"].map(FACILITY_STAGE)
usgs_mask = combined["SOURCE"] == "USGS_MINFAC_2017"

dupes = []
for _, srow in combined[serna_mask & (combined["_stage"] == "extraction")].iterrows():
    pot = combined[usgs_mask & (combined["_name_lower"] == srow["_name_lower"])
                   & (combined["PRIMARY_COMMODITY"] == srow["PRIMARY_COMMODITY"])]
    dupes.extend(pot.index.tolist())
dupes = list(set(dupes))
if dupes:
    combined.drop(index=dupes, inplace=True)
    print(f"Dropped {len(dupes)} USGS duplicates")

combined.drop(columns=["_name_lower", "_stage"], errors="ignore", inplace=True)
combined.reset_index(drop=True, inplace=True)
print(f"After dedup: {len(combined)}")

# %% 5. Parse secondary commodities
combined["COMMODITY_LIST"] = combined["ALL_COMMODITIES_RAW"].apply(parse_commodity_list)
combined["N_COMMODITIES"] = combined["COMMODITY_LIST"].apply(len)

multi = combined[combined["N_COMMODITIES"] > 1]
print(f"Multi-commodity facilities: {len(multi)}, max per facility: {combined['N_COMMODITIES'].max()}")

exploded = combined.explode("COMMODITY_LIST").reset_index(drop=True)
exploded = exploded.drop(columns=["COMMODITY"], errors="ignore")
exploded = exploded.rename(columns={"COMMODITY_LIST": "COMMODITY"})
exploded = exploded[exploded["COMMODITY"].notna() & (exploded["COMMODITY"].astype(str) != "")].reset_index(drop=True)

commodity_counts = exploded.groupby("COMMODITY")["FACILITY_NAME"].count().sort_values(ascending=False)
print(f"Unique commodities: {len(commodity_counts)}")
print(commodity_counts.head(15).to_string())

# %% 6. Geographic consistency checks
has_coords = combined[combined["LATITUD"].notna() & combined["LONGITUD"].notna()].copy()
print(f"Records with coordinates: {len(has_coords)} / {len(combined)}")

outside = has_coords[
    (has_coords["LONGITUD"] > CHILE_LON_MAX) | (has_coords["LONGITUD"] < CHILE_LON_MIN) |
    (has_coords["LATITUD"] > CHILE_LAT_MAX) | (has_coords["LATITUD"] < CHILE_LAT_MIN)]
if len(outside) > 0:
    print(f"{len(outside)} records OUTSIDE Chile bounding box:")
    print(outside[["FACILITY_NAME", "REGION", "LONGITUD", "LATITUD", "PRIMARY_COMMODITY", "SOURCE"]].to_string(index=False))
else:
    print("All records within Chile bounding box.")

border = has_coords[has_coords["LONGITUD"] > -67.5].sort_values("LONGITUD", ascending=False)
print(f"Border-zone records (lon > -67.5): {len(border)}")
if len(border) > 0:
    print(border[["FACILITY_NAME", "REGION", "LONGITUD", "LATITUD", "PRIMARY_COMMODITY"]].to_string(index=False))

# %% 7. Link mines to processing facilities
combined["_stage"] = combined["FACILITY_TYPE"].map(FACILITY_STAGE)

mines = combined[(combined["_stage"] == "extraction")
                 & combined["LATITUD"].notna() & combined["LONGITUD"].notna()].copy()
plants_link = combined[(combined["_stage"] == "processing")
                       & combined["LATITUD"].notna() & combined["LONGITUD"].notna()].copy()

print(f"Mines/prospects with coords: {len(mines)}")
print(f"Processing facilities with coords: {len(plants_link)}")

mine_comms = {idx: set(r.get("COMMODITY_LIST") or [r["PRIMARY_COMMODITY"]]) for idx, r in mines.iterrows()}
plant_comms = {idx: set(r.get("COMMODITY_LIST") or [r["PRIMARY_COMMODITY"]]) for idx, r in plants_link.iterrows()}

link_rows = []
for midx, mrow in mines.iterrows():
    mc = mine_comms[midx]
    if not mc:
        continue
    for pidx, prow in plants_link.iterrows():
        shared = mc & plant_comms[pidx]
        if not shared:
            continue
        dist = haversine_km(mrow["LATITUD"], mrow["LONGITUD"], prow["LATITUD"], prow["LONGITUD"])
        if dist <= LINK_MAX_KM:
            link_rows.append({
                "MINE_NAME": mrow["FACILITY_NAME"], "MINE_IDX": midx,
                "MINE_TYPE": mrow["FACILITY_TYPE"], "MINE_STATUS": mrow["STATUS"],
                "MINE_LAT": mrow["LATITUD"], "MINE_LON": mrow["LONGITUD"],
                "MINE_REGION": mrow.get("REGION"), "MINE_OPERATOR": mrow.get("OPERATOR_NAME"),
                "PLANT_NAME": prow["FACILITY_NAME"], "PLANT_IDX": pidx,
                "PLANT_TYPE": prow["FACILITY_TYPE"], "PLANT_STATUS": prow["STATUS"],
                "PLANT_LAT": prow["LATITUD"], "PLANT_LON": prow["LONGITUD"],
                "PLANT_OPERATOR": prow.get("OPERATOR_NAME"), "PLANT_OWNER": prow.get("OWNER_NAME"),
                "PLANT_CAPACITY": prow.get("CAPACITY"), "PLANT_CAPACITY_UNITS": prow.get("CAPACITY_UNITS"),
                "SHARED_COMMODITIES": ", ".join(sorted(shared)),
                "DISTANCE_KM": round(dist, 1),
            })

links_df = pd.DataFrame(link_rows)
if len(links_df) > 0:
    links_df.sort_values(["MINE_NAME", "DISTANCE_KM"], inplace=True)
    print(f"Links: {len(links_df)}, unique mines: {links_df['MINE_NAME'].nunique()}, "
          f"unique plants: {links_df['PLANT_NAME'].nunique()}")
    unlinked = mines[~mines.index.isin(set(links_df['MINE_IDX']))]
    print(f"Mines with no nearby processing: {len(unlinked)}")
    if len(unlinked) > 0:
        print(unlinked["PRIMARY_COMMODITY"].value_counts().to_string())
else:
    print("No links found within distance threshold.")

combined.drop(columns=["_stage"], errors="ignore", inplace=True)

# %% 8. Trade data cross-reference
HS92_COMMODITY_MAP = {
    "Copper": {"260300": "Copper ores and concentrates", "740311": "Refined copper, cathodes",
               "740312": "Refined copper, wire bars", "740400": "Copper waste and scrap",
               "740319": "Refined copper, other", "740321": "Copper alloys, copper-zinc",
               "740329": "Copper alloys, other"},
    "Lithium": {"283691": "Lithium carbonate", "253090": "Lithium minerals (ores)"},
    "Molybdenum": {"261310": "Molybdenum ores, roasted", "261390": "Molybdenum ores, other",
                   "810294": "Molybdenum, unwrought"},
    "Gold": {"710812": "Gold, non-monetary, unwrought", "261690": "Precious metal ores"},
    "Silver": {"710691": "Silver, unwrought", "261610": "Silver ores and concentrates"},
    "Iron": {"260111": "Iron ores, non-agglomerated", "260112": "Iron ores, agglomerated",
             "720110": "Pig iron", "720241": "Ferrochromium"},
    "Nitrate": {"310239": "Sodium nitrate", "310250": "Sodium nitrate, natural", "283410": "Nitrites"},
    "Iodine": {"280120": "Iodine"},
    "Boron": {"252810": "Borates, natural", "281000": "Boric acid"},
    "Zinc": {"260800": "Zinc ores and concentrates", "790111": "Zinc, not alloyed, unwrought"},
    "Rhenium": {"811292": "Rhenium, unwrought"},
    "Potassium": {"310420": "Potassium chloride"},
    "Rare Earths": {"280530": "Rare earth metals"},
    "Selenium": {"280490": "Selenium"},
}

summary = None
if os.path.exists(HS92_PATH):
    trade = pd.read_csv(HS92_PATH)
    cols = trade.columns.tolist()
    product_col = "product_hs92_code"
    export_col = next((c for c in cols if "export" in c.lower()), None)
    year_col = next((c for c in cols if "year" in c.lower()), None)

    if all([product_col in cols, export_col, year_col]):
        trade[product_col] = trade[product_col].astype(str).str.zfill(6)
        latest_year = trade[year_col].max()
        print(f"Trade data: {trade.shape}, latest year: {latest_year}")

        summary_rows = []
        for commodity, hs_codes in HS92_COMMODITY_MAP.items():
            n_active = len(combined[(combined["FACILITY_TYPE"] == "Mine (active)")
                                    & combined["COMMODITY_LIST"].apply(lambda x: commodity in (x or []))])
            n_mines = len(combined[combined["FACILITY_TYPE"].isin(["Mine (active)", "Mine (idle)", "Prospect/Project"])
                                   & combined["COMMODITY_LIST"].apply(lambda x: commodity in (x or []))])
            n_plants = len(combined[(combined["FACILITY_TYPE"].map(FACILITY_STAGE) == "processing")
                                    & combined["COMMODITY_LIST"].apply(lambda x: commodity in (x or []))])
            n_links = len(links_df[links_df["SHARED_COMMODITIES"].str.contains(commodity, na=False)]) if len(links_df) else 0

            hs_list = list(hs_codes.keys())
            lt = trade[(trade[year_col] == latest_year) & (trade[product_col].isin(hs_list))]
            total_exp = lt[export_col].sum() if len(lt) else 0

            raw_codes = [c for c in hs_list if any(k in hs_codes[c].lower()
                         for k in ["ore", "concentrate", "mineral", "natural"])]
            proc_codes = [c for c in hs_list if c not in raw_codes]
            raw_exp = trade[(trade[year_col] == latest_year) & (trade[product_col].isin(raw_codes))][export_col].sum() if raw_codes else 0
            proc_exp = trade[(trade[year_col] == latest_year) & (trade[product_col].isin(proc_codes))][export_col].sum() if proc_codes else 0

            if n_active > 0 and n_plants > 0:
                status = "Extraction + Processing"
            elif n_active > 0:
                status = "Extraction only"
            elif n_plants > 0:
                status = "Processing only"
            else:
                status = "No active facilities"

            summary_rows.append({
                "COMMODITY": commodity, "ACTIVE_MINES": n_active,
                "TOTAL_MINES_PROSPECTS": n_mines, "PROCESSING_FACILITIES": n_plants,
                "MINE_PLANT_LINKS": n_links, "CHAIN_STATUS": status,
                f"EXPORT_TOTAL_{latest_year}_USD": total_exp,
                f"EXPORT_RAW_{latest_year}_USD": raw_exp,
                f"EXPORT_PROCESSED_{latest_year}_USD": proc_exp,
                "RAW_SHARE": round(raw_exp / total_exp, 3) if total_exp > 0 else None,
            })
        summary = pd.DataFrame(summary_rows).sort_values(f"EXPORT_TOTAL_{latest_year}_USD", ascending=False)
        print(summary.to_string(index=False))
else:
    print(f"HS92 file not found at {HS92_PATH}")

# %% 9. COCHILCO production cross-reference
#
# Loads the extracted COCHILCO yearbook and checks pipeline facility counts and
# commodity coverage against official national and regional production totals.

cochilco_checks = {}

if os.path.exists(COCHILCO_PATH):
    print("Loading COCHILCO yearbook...")

    # --- A: National production totals (latest year) ---
    nat = pd.read_excel(COCHILCO_PATH, sheet_name="A_National_Production",
                        header=3, index_col=0)
    nat.columns = [int(c) if isinstance(c, (int, float)) else c for c in nat.columns]

    # Map COCHILCO row labels to pipeline commodity names
    COCHILCO_NAT_MAP = {
        "Copper":      "COBRE (Miles de TM de fino) / Copper (kMT Fine Content)",
        "Molybdenum":  "MOLIBDENO (TM de fino) / Molybdenum (MT Fine Content)",
        "Gold":        "ORO (Kg de fino) / Gold (Kg Fine Content)",
        "Silver":      "PLATA (Kg de fino) / Silver (Kg Fine Content)",
        "Iron":        "HIERRO (Miles de TM de fino) / Iron (kMT Fine Content)",
        "Zinc":        "ZINC (TM de fino) / Zinc (MT Fine Content)",
    }

    latest_yr = max(c for c in nat.columns if isinstance(c, int))
    print(f"\nCOCHILCO latest year: {latest_yr}")
    print(f"{'Commodity':<15} {'COCHILCO Prod':>15} {'Pipeline Mines':>15} {'Pipeline Plants':>15}")
    print("-" * 62)

    for comm, row_label in COCHILCO_NAT_MAP.items():
        prod = nat.loc[row_label, latest_yr] if row_label in nat.index else None
        n_mines = len(combined[(combined["FACILITY_TYPE"].isin(["Mine (active)", "Mine (idle)"]))
                               & combined["COMMODITY_LIST"].apply(lambda x: comm in (x or []))])
        n_plants = len(combined[(combined["FACILITY_TYPE"].map(FACILITY_STAGE) == "processing")
                                & combined["COMMODITY_LIST"].apply(lambda x: comm in (x or []))])
        prod_str = f"{prod:,.1f}" if prod is not None else "N/A"
        print(f"{comm:<15} {prod_str:>15} {n_mines:>15} {n_plants:>15}")
        cochilco_checks[comm] = {"production": prod, "pipeline_mines": n_mines, "pipeline_plants": n_plants}

    # --- B: Company-level copper check ---
    cu_co = pd.read_excel(COCHILCO_PATH, sheet_name="B1_Copper_by_Company",
                          header=3, index_col=0)
    cu_co.columns = [int(c) if isinstance(c, (int, float)) else c for c in cu_co.columns]

    print(f"\nTop copper producers ({latest_yr}, kMT fine Cu):")
    for label in cu_co.index:
        val = cu_co.loc[label, latest_yr] if latest_yr in cu_co.columns else None
        if val is not None and isinstance(val, (int, float)) and val > 50:
            # Check if this company name appears in pipeline
            name_lower = str(label).lower()
            matched = combined[combined["FACILITY_NAME"].str.lower().str.contains(
                name_lower[:10], na=False, regex=False)]
            status = f"matched ({len(matched)})" if len(matched) > 0 else "NOT IN PIPELINE"
            print(f"  {label:<40} {val:>10,.1f}  {status}")

    # --- C: Regional production check (Antofagasta as example) ---
    for region_sheet, region_name in [("C_Antofagasta", "Antofagasta"),
                                       ("C_Atacama", "Atacama"),
                                       ("C_Coquimbo", "Coquimbo")]:
        reg = pd.read_excel(COCHILCO_PATH, sheet_name=region_sheet,
                            header=3, index_col=0)
        reg.columns = [int(c) if isinstance(c, (int, float)) else c for c in reg.columns]

        cu_row = [r for r in reg.index if "COBRE" in str(r)]
        if cu_row and latest_yr in reg.columns:
            cu_val = reg.loc[cu_row[0], latest_yr]
            # Count pipeline mines in this region
            region_mines = combined[
                combined["REGION"].str.contains(region_name, case=False, na=False)
                & combined["COMMODITY_LIST"].apply(lambda x: "Copper" in (x or []))
                & combined["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
            ]
            print(f"\n{region_name}: COCHILCO Cu = {cu_val:,.0f} MT, pipeline mines = {len(region_mines)}")

    # --- Non-metallic: lithium compounds check ---
    nat_litio = [r for r in nat.index if "LITIO" in str(r).upper() and "CARBONATO" in str(r).upper()]
    if nat_litio:
        li_prod = nat.loc[nat_litio[0], latest_yr]
        li_mines = len(combined[combined["COMMODITY_LIST"].apply(lambda x: "Lithium" in (x or []))
                                & combined["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)])
        print(f"\nLithium carbonate: COCHILCO = {li_prod:,.0f} MT, pipeline Li mines = {li_mines}")

else:
    print(f"COCHILCO file not found at {COCHILCO_PATH}")

# %% 10. Save pipeline outputs
drop_cols = ["_stage", "_name_lower", "COMMODITY_LIST"]
out = combined.drop(columns=[c for c in drop_cols if c in combined.columns], errors="ignore")
if "COMMODITY_LIST" in combined.columns:
    out["COMMODITY_LIST_STR"] = combined["COMMODITY_LIST"].apply(
        lambda x: ", ".join(x) if isinstance(x, list) else x)

out.to_file(os.path.join(DIR_PRELIM, "Chile_Minerals_Inventory.geojson"), driver="GeoJSON")
pd.DataFrame(out.drop(columns="geometry")).to_csv(
    os.path.join(DIR_PRELIM, "Chile_Minerals_Inventory.csv"), index=False)

if len(links_df) > 0:
    links_df.to_csv(os.path.join(DIR_PRELIM, "Chile_Mine_Plant_Links.csv"), index=False)
if summary is not None:
    summary.to_csv(os.path.join(DIR_PRELIM, "Chile_Supply_Chain_Summary.csv"), index=False)

elapsed = time.time() - start_time
print(f"\nPipeline complete in {elapsed:.1f}s")
print(f"  Inventory: {len(out)} records -> {DIR_PRELIM}")
print(f"  Links: {len(links_df)}")
if cochilco_checks:
    print(f"  COCHILCO cross-ref: {len(cochilco_checks)} commodities checked")

Deposits: 267, Reserve/resource records: 365
Sernageomin: (267, 65)
USGS Chile: 394 total, 194 plants, 192 mines
Before dedup: 461
Enriched 62 records with USGS ownership
After dedup: 461
Multi-commodity facilities: 142, max per facility: 4
Unique commodities: 44
COMMODITY
Copper               193
Gold                 103
Silver                63
Molybdenum            44
Iron                  26
Nitrate               21
Iodine                21
Boron                 20
Calcium Carbonate     16
Cobalt                16
Lithium               15
Zinc                  14
Rhenium               14
Silica                10
Potassium              9
Records with coordinates: 461 / 461
All records within Chile bounding box.
Border-zone records (lon > -67.5): 5
              FACILITY_NAME                               REGION   LONGITUD    LATITUD PRIMARY_COMMODITY
       Lavadero Isla Picton Magallanes y de la Antártica Chilena -66.950000 -55.033333              Gold
         Salar de Quisquiro  